In [1]:
import json
import math
import os
import functions_for_experiments
from openai import OpenAI
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient, models
from qdrant_client.models import SparseTextEmbedding
from thefuzz import fuzz
from fastembed import SparseTextEmbedding

c:\Users\Olena\Downloads\ucu-employee-support-rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7196.36it/s]


In [2]:
with open("final_golden_dataset copy.json", "r", encoding="utf-8") as f:
    golden_data = json.load(f)

In [3]:
n = len(golden_data)

## Baseline

In [4]:
MODEL_E5_LARGE = SentenceTransformer('intfloat/multilingual-e5-large')
COLLECTION_E5_LARGE = "ucu_documents_e5_large"

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3061.51it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
total_hit_e5_large, total_mrr_e5_large, total_recall_e5_large, total_ndcg_e5_large, \
total_hit_reranked_e5_large, total_mrr_reranked_e5_large, total_recall_reranked_e5_large, total_ndcg_reranked_e5_large = functions_for_experiments.get_metrics(MODEL_E5_LARGE, COLLECTION_E5_LARGE, e5=True)

In [6]:
print("Results for E5 large model:")
print(f"Hit Rate @ 5: {round(total_hit_e5_large/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_e5_large/n, 2)}")
print(f"Recall @ 5: {round(total_recall_e5_large/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_e5_large/n, 2)}")

Results for E5 large model:
Hit Rate @ 5: 0.8
MRR @ 5: 0.62
Recall @ 5: 0.71
NDCG @ 5: 0.62


In [7]:
print("Results for E5 large model:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_large/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_large/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_large/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_large/n, 2)}")

Results for E5 large model:
Hit Rate @ 5: 0.82
MRR @ 5: 0.75
Recall @ 5: 0.76
NDCG @ 5: 0.72


## HyDE

In [5]:
total_hit_e5_large_hyde, total_mrr_e5_large_hyde, total_recall_e5_large_hyde, total_ndcg_e5_large_hyde, \
total_hit_reranked_e5_large_hyde, total_mrr_reranked_e5_large_hyde, total_recall_reranked_e5_large_hyde, total_ndcg_reranked_e5_large_hyde = functions_for_experiments.get_metrics_hyde(MODEL_E5_LARGE, COLLECTION_E5_LARGE)

In [6]:
print("Results for E5 large model (HyDE):")
print(f"Hit Rate @ 5: {round(total_hit_e5_large_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_e5_large_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_e5_large_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_e5_large_hyde/n, 2)}")

Results for E5 large model (HyDE):
Hit Rate @ 5: 0.36
MRR @ 5: 0.27
Recall @ 5: 0.29
NDCG @ 5: 0.25


In [7]:
print("Results for E5 large model (HyDE):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_large_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_large_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_large_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_large_hyde/n, 2)}")

Results for E5 large model (HyDE):
Hit Rate @ 5: 0.5
MRR @ 5: 0.45
Recall @ 5: 0.42
NDCG @ 5: 0.4


## Query Transform

In [8]:
total_hit_e5_large_transform, total_mrr_e5_large_transform, total_recall_e5_large_transform, total_ndcg_e5_large_transform, \
total_hit_reranked_e5_large_transform, total_mrr_reranked_e5_large_transform, total_recall_reranked_e5_large_transform, total_ndcg_reranked_e5_large_transform = functions_for_experiments.get_metrics_query_transform(MODEL_E5_LARGE, COLLECTION_E5_LARGE, e5=True)

In [9]:
print("Results for E5 large model (Query Transform):")
print(f"Hit Rate @ 5: {round(total_hit_e5_large_transform/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_e5_large_transform/n, 2)}")
print(f"Recall @ 5: {round(total_recall_e5_large_transform/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_e5_large_transform/n, 2)}")

Results for E5 large model (Query Transform):
Hit Rate @ 5: 0.78
MRR @ 5: 0.52
Recall @ 5: 0.7
NDCG @ 5: 0.54


In [10]:
print("Results for E5 large model (Query Transform):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_large_transform/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_large_transform/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_large_transform/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_large_transform/n, 2)}")

Results for E5 large model (Query Transform):
Hit Rate @ 5: 0.86
MRR @ 5: 0.79
Recall @ 5: 0.78
NDCG @ 5: 0.74


## Hybrid

In [5]:
MODEL_SPARSE = SparseTextEmbedding(model_name="Qdrant/bm25")

In [ ]:
total_hit_e5_large_sparse, total_mrr_e5_large_sparse, total_recall_e5_large_sparse, total_ndcg_e5_large_sparse, \
total_hit_reranked_e5_large_sparse, total_mrr_reranked_e5_large_sparse, total_recall_reranked_e5_large_sparse, total_ndcg_reranked_e5_large_sparse = functions_for_experiments.get_metrics_sparse(MODEL_E5_LARGE, COLLECTION_E5_LARGE, sparse_model=MODEL_SPARSE, e5=True)

In [16]:
print("Results for E5 large model (Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_e5_large_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_e5_large_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_e5_large_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_e5_large_sparse/n, 2)}")

Results for E5 large model (Hybrid):
Hit Rate @ 5: 0.78
MRR @ 5: 0.56
Recall @ 5: 0.69
NDCG @ 5: 0.56


In [17]:
print("Results for E5 large model (Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_large_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_large_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_large_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_large_sparse/n, 2)}")

Results for E5 large model (Hybrid):
Hit Rate @ 5: 0.84
MRR @ 5: 0.75
Recall @ 5: 0.76
NDCG @ 5: 0.72


## Query Transform + Hybrid

In [6]:
total_hit_e5_large_transform_hybrid, total_mrr_e5_large_transform_hybrid, total_recall_e5_large_transform_hybrid, total_ndcg_e5_large_transform_hybrid, \
total_hit_reranked_e5_large_transform_hybrid, total_mrr_reranked_e5_large_transform_hybrid, total_recall_reranked_e5_large_transform_hybrid, total_ndcg_reranked_e5_large_transform_hybrid = functions_for_experiments.get_metrics_query_transform(MODEL_E5_LARGE, COLLECTION_E5_LARGE, e5=True, sparse_model=MODEL_SPARSE)

In [7]:
print("Results for E5 large model (Query Transform + Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_e5_large_transform_hybrid/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_e5_large_transform_hybrid/n, 2)}")
print(f"Recall @ 5: {round(total_recall_e5_large_transform_hybrid/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_e5_large_transform_hybrid/n, 2)}")

Results for E5 large model (Query Transform + Hybrid):
Hit Rate @ 5: 0.72
MRR @ 5: 0.55
Recall @ 5: 0.65
NDCG @ 5: 0.55


In [8]:
print("Results for E5 large model (Query Transform + Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_e5_large_transform_hybrid/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_e5_large_transform_hybrid/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_e5_large_transform_hybrid/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_e5_large_transform_hybrid/n, 2)}")

Results for E5 large model (Query Transform + Hybrid):
Hit Rate @ 5: 0.86
MRR @ 5: 0.76
Recall @ 5: 0.76
NDCG @ 5: 0.72
